# Iterative Workflows in LangGraph

## 1. What is an Iterative Workflow?

An **Iterative Workflow** is a workflow where one or more nodes are executed **repeatedly until a specific condition is satisfied**.

### Simple Definition

> An Iterative Workflow repeatedly performs a task, checks the result, and either repeats the task or moves forward.

### Basic Structure

    START
      ↓
    Task
      ↓
    Check Condition
      ↓
    ┌───────────────┐
    │               │
    │   Continue?   │
    │               │
    └──────┬────────┘
       Yes │   No
           │
           ↓
         Task
           ↑
           │
           └──────────────

                    No
                    ↓
                   END


The important idea is:

    Task
      ↓
    Check
      ↓
    Repeat if needed
      ↓
    Stop when condition is satisfied


---

# 2. Why Do We Need Iterative Workflows?

Many real-world AI tasks cannot be completed correctly in a single attempt.

For example:

    Generate Answer
          ↓
       Validate
          ↓
       Correct?
       /     \
     No       Yes
     ↓         ↓
  Improve     END
     ↓
  Generate
     ↑
     └────────

The system keeps improving the answer until it passes validation.

---

# 3. Sequential vs Conditional vs Iterative

We have now learned three important workflow patterns.

## Sequential Workflow

    A → B → C → END

The workflow moves forward once.

> Fixed sequence.


## Conditional Workflow

    A
    ↓
    Router
    /   \
   B     C

The workflow chooses a path.

> Dynamic decision.


## Iterative Workflow

    A
    ↓
    B
    ↓
    Condition
    ↓
   Yes → A
    │
   No
    ↓
   END

The workflow can go backward and repeat.

> Repeated execution until a stopping condition is met.


---

# 4. The Core Idea of Iteration

The simplest mental model is:

    DO
      ↓
    CHECK
      ↓
    DONE?
     /  \
   No    Yes
   ↓      ↓
  DO     END
   ↑
   └──────

This is similar to a Python `while` loop.

Python:

    while not condition:
        do_task()


LangGraph:

    START
      ↓
    do_task
      ↓
    check_condition
      ↓
    ┌───────────────┐
    ↓               ↓
   retry            END
    ↓
    └──→ do_task


---

# 5. Iteration in LangGraph

LangGraph allows us to create cycles in a graph.

Normally:

    A → B → C → END

An iterative graph can be:

    A → B → A

or:

    A → B → C → B

The graph continues executing until a condition routes execution to `END`.

### Important

An iterative workflow requires:

1. A task
2. A condition/check
3. A route back to the task
4. A termination condition


---

# 6. Simple Real-World Example

Imagine we want an AI system to improve a number until it reaches 10.

Start:

    number = 0

Each iteration:

    number += 1

Check:

    Is number >= 10?

If:

    No → increment again

If:

    Yes → END

Flow:

    START
      ↓
    Increment
      ↓
    Check
     /   \
   No     Yes
   ↓       ↓
Increment  END
   ↑
   └───────


---

# 7. Simple LangGraph Coding Example

This is a very simple example to understand the concept.

    from typing import TypedDict
    from langgraph.graph import StateGraph, START, END


    class State(TypedDict):
        number: int


    def increment(state: State):
        return {
            "number": state["number"] + 1
        }


    def should_continue(state: State):
        if state["number"] < 5:
            return "continue"

        return "stop"


    graph = StateGraph(State)

    graph.add_node("increment", increment)

    graph.add_edge(START, "increment")

    graph.add_conditional_edges(
        "increment",
        should_continue,
        {
            "continue": "increment",
            "stop": END
        }
    )

    app = graph.compile()

    result = app.invoke({
        "number": 0
    })

    print(result)


### Output

    {'number': 5}


---

# 8. Understanding the Code

Let's understand it step by step.

## Step 1 — Define State

    class State(TypedDict):
        number: int

The state contains:

    number

Initially:

    number = 0


---

## Step 2 — Create the Iterative Node

    def increment(state: State):
        return {
            "number": state["number"] + 1
        }

Every time this node runs:

    0 → 1
    1 → 2
    2 → 3
    3 → 4
    4 → 5


---

## Step 3 — Create the Condition

    def should_continue(state: State):
        if state["number"] < 5:
            return "continue"

        return "stop"


This function decides:

    number < 5
         ↓
      continue

    number >= 5
         ↓
       stop


---

## Step 4 — Add Conditional Edges

    graph.add_conditional_edges(
        "increment",
        should_continue,
        {
            "continue": "increment",
            "stop": END
        }
    )

This is the most important part.

It means:

    increment
        ↓
    should_continue()
        ↓
    ┌─────────────┐
    ↓             ↓
 continue        stop
    ↓             ↓
increment        END


Notice:

    "continue": "increment"

The node points back to itself.

That creates the loop.


---

# 9. Complete Execution

Initial state:

    number = 0

### Iteration 1

    increment

    0 → 1

Check:

    1 < 5

Result:

    continue


### Iteration 2

    increment

    1 → 2

Check:

    2 < 5

Result:

    continue


### Iteration 3

    increment

    2 → 3

Check:

    3 < 5

Result:

    continue


### Iteration 4

    increment

    3 → 4

Check:

    4 < 5

Result:

    continue


### Iteration 5

    increment

    4 → 5

Check:

    5 < 5 → False

Result:

    stop


Therefore:

    END


Final state:

    number = 5


---

# 10. Graph Visualization

The graph is:

                ┌──────────────┐
                │              ↓
    START → Increment → Condition
                ↑          /    \
                │         /      \
                └──continue      stop
                                  ↓
                                 END


This is the basic structure of an iterative workflow.


---

# 11. Iterative Workflow = Loop

You can think about LangGraph iteration as a graph version of a Python loop.

Python:

    number = 0

    while number < 5:
        number += 1

LangGraph:

    START
      ↓
    increment
      ↓
    condition
      ↓
    ┌───────────┐
    ↓           ↓
  repeat       END
    │
    └──→ increment


The difference is that LangGraph represents the loop as **nodes + edges + state**.


---

# 12. Why Use LangGraph Instead of a Normal Python Loop?

For a simple counter, you should obviously use a normal Python loop.

LangGraph becomes useful when the repeated operation is more complex.

For example:

    Generate Answer
         ↓
    Validate Answer
         ↓
    Good?
      /   \
    No     Yes
    ↓       ↓
  Improve  END
    ↓
 Generate Answer


Here each iteration may involve:

- LLM calls
- Tools
- Retrieval
- Validation
- State updates
- Human approval
- Error handling

LangGraph gives structure and control over this process.


---

# 13. Iterative LLM Workflow

A common AI pattern is:

    Generate
       ↓
    Evaluate
       ↓
    Good enough?
      /      \
    No        Yes
    ↓          ↓
  Improve     END
    ↓
  Generate
    ↑
    └────────


Example:

    User:
    "Write a professional email."

The system can:

    Generate Email
          ↓
    Evaluate Quality
          ↓
    Quality >= 80%?
       /       \
     No         Yes
     ↓           ↓
   Improve      END
     ↓
   Generate
     ↑
     └────────


This is an **iterative refinement workflow**.


---

# 14. Iterative Workflow for LLM Output Refinement

A more realistic architecture:

    START
      ↓
    Generate Draft
      ↓
    Evaluate Draft
      ↓
    ┌────────────────┐
    │ Score >= 8 ?   │
    └───────┬────────┘
        No  │  Yes
        ↓   │   ↓
     Improve│  END
        ↓   │
     Generate
        ↑
        └──────


The state could contain:

    {
        "draft": "...",
        "score": 6,
        "attempts": 1
    }

After improvement:

    {
        "draft": "better draft",
        "score": 8,
        "attempts": 2
    }


---

# 15. Iteration with Maximum Attempts

Never assume an iterative workflow will always reach the desired condition.

For example:

    Generate
       ↓
    Validate
       ↓
    Valid?
     /   \
   Yes    No
   ↓       ↓
  END    attempts < 3?
          /       \
        Yes        No
         ↓          ↓
      Retry      Fallback
         ↓          ↓
      Generate     END


State:

    class State(TypedDict):
        output: str
        attempts: int
        valid: bool


The router can check:

    if state["valid"]:
        return "done"

    if state["attempts"] < 3:
        return "retry"

    return "fallback"


This prevents infinite loops.


---

# 16. Why Maximum Attempts Matter

Suppose:

    Generate → Validate → Invalid → Generate → Validate → Invalid → ...

If there is no stopping condition, the workflow could theoretically continue forever.

Therefore production systems should often have:

    Maximum attempts
    OR
    Maximum execution time
    OR
    Maximum token/cost budget
    OR
    Another termination condition


---

# 17. Iterative Workflow with Attempts

Example:

    START
      ↓
    Generate
      ↓
    attempts += 1
      ↓
    Validate
      ↓
    ┌───────────────────┐
    ↓                   ↓
   Valid              Invalid
    ↓                   ↓
   END            attempts < 3?
                       /   \
                     Yes    No
                      ↓      ↓
                   Generate Fallback
                      ↑       ↓
                      └──────END


State evolution:

    attempts = 1
        ↓
    attempts = 2
        ↓
    attempts = 3
        ↓
    END


---

# 18. Iterative Agent Workflow

Iteration is one of the most important concepts in Agentic AI.

An agent usually works like this:

    Goal
      ↓
    Observe
      ↓
    Decide
      ↓
    Act
      ↓
    Observe Result
      ↓
    Goal achieved?
      /       \
    No         Yes
    ↓           ↓
  Decide       END
    ↑
    └───────────


This is an iterative loop.


---

# 19. Agentic Loop

The famous agent loop can be represented as:

    ┌──────────────────────────────┐
    │                              ↓
    │    Observe → Reason → Act → Observe
    │                              │
    │                              │
    └──────── Continue? ───────────┘
                     │
                     ↓
                    END


More explicitly:

    User Goal
       ↓
    Agent
       ↓
    Select Action
       ↓
    Tool
       ↓
    Tool Result
       ↓
    Agent
       ↓
    Need another action?
      /          \
    Yes           No
    ↓              ↓
  Tool           Answer
    ↓              ↓
    └──────→       END


The agent repeats until it decides that the goal is complete.


---

# 20. Iterative RAG

Iteration can also be used in RAG.

Example:

    User Query
        ↓
    Retrieve Documents
        ↓
    Evaluate Documents
        ↓
    Relevant?
      /    \
    Yes     No
    ↓        ↓
 Generate   Rewrite Query
    ↓        ↓
   END    Retrieve Again
             ↑
             └────────


This is sometimes called **iterative retrieval** or **query refinement**.

The system can improve the query when the retrieved documents are insufficient.


---

# 21. Iterative RAG Example

Initial query:

    "How can I improve model performance?"

Retriever returns poor results.

The evaluator determines:

    relevance = low

Router:

    low relevance
        ↓
    Rewrite Query
        ↓
    "Techniques to improve machine learning model generalization"
        ↓
    Retrieve Again
        ↓
    Evaluate
        ↓
    relevance = high
        ↓
    Generate Answer
        ↓
       END


This is a powerful Agentic RAG pattern.


---

# 22. Iterative Workflow for Code Generation

Another example:

    User Request
         ↓
    Generate Code
         ↓
    Run Tests
         ↓
    Tests Passed?
       /       \
     Yes        No
      ↓          ↓
     END       Analyze Error
                 ↓
             Fix Code
                 ↓
            Generate/Update
                 ↑
                 └────────


This is commonly used in coding agents.

The system repeatedly:

    Generate
    → Test
    → Analyze
    → Fix
    → Test again

until the code passes or a retry limit is reached.


---

# 23. Iterative Workflow for SQL

AI Data Analyst example:

    User Query
        ↓
    Generate SQL
        ↓
    Validate SQL
        ↓
    Valid?
      /    \
    Yes     No
    ↓        ↓
  Execute   Fix SQL
    ↓        ↓
  Success?  Generate Again
   /   \
 Yes    No
 ↓       ↓
Analyze  Fix SQL
 ↓       ↑
END     └──────


This combines:

- Sequential workflow
- Conditional workflow
- Iterative workflow


---

# 24. Iterative Workflow in AI Travel Planner

An AI Travel Planner could iteratively improve an itinerary.

Example:

    User Requirements
          ↓
    Generate Itinerary
          ↓
    Validate
          ↓
    Meets requirements?
       /          \
     Yes           No
      ↓             ↓
    END          Improve
                    ↓
             Generate Again
                    ↑
                    └──────


Validation might check:

    Budget
    Number of days
    Destination
    Opening hours
    Travel distance
    User preferences

If requirements are not satisfied, the planner improves the itinerary.


---

# 25. State Is Critical in Iterative Workflows

The state must preserve information between iterations.

Example:

    class State(TypedDict):
        draft: str
        score: int
        attempts: int


First iteration:

    draft = "Initial draft"
    score = 5
    attempts = 1


Second iteration:

    draft = "Improved draft"
    score = 7
    attempts = 2


Third iteration:

    draft = "Final draft"
    score = 9
    attempts = 3


The state acts as the memory of the current workflow execution.


---

# 26. What Should State Store?

Depending on the workflow, state may contain:

    Current output
    Previous output
    Validation result
    Score
    Attempt count
    Error information
    Tool results
    Retrieved documents
    User requirements
    Completion status


For example:

    class State(TypedDict):
        query: str
        output: str
        score: float
        attempts: int
        feedback: str
        done: bool


---

# 27. Iteration and State Evolution

Think of state as changing after every loop.

    Initial State
         ↓
    Iteration 1
         ↓
    Updated State
         ↓
    Iteration 2
         ↓
    Updated State
         ↓
    Iteration 3
         ↓
    Final State


Example:

    score = 4
       ↓
    score = 6
       ↓
    score = 8
       ↓
    score = 9
       ↓
    END


---

# 28. Iteration and Conditional Edges

A key insight:

> **Iterative workflows are usually built using conditional edges.**

Why?

Because after every iteration the graph needs to answer:

    Continue?
        OR
    Stop?


So:

    Task
      ↓
    Conditional Router
      ↓
    ┌────────────┬────────────┐
    ↓            ↓
   Retry        END
    ↓
   Task


Therefore:

    Iteration
    =
    Cycle
    +
    Condition


---

# 29. Self-Loop

A very simple iterative graph uses a self-loop.

Example:

    ┌──────────────┐
    │              ↓
    START → Task → Router
               ↑     │
               │     │
               └─────┘
                  retry

The node can route back to itself.

Example:

    "retry": "task"

This is called a **self-loop**.


---

# 30. Multi-Node Iteration

Iteration does not have to repeat only one node.

Example:

    Generate
       ↓
    Validate
       ↓
    Improve?
     /     \
   Yes      No
   ↓         ↓
 Generate   END


Or:

    Generate
       ↓
    Test
       ↓
    Analyze
       ↓
    Fix
       ↓
    Test
       ↑
       └────────────


The entire sequence can form a cycle.


---

# 31. Iterative Workflow with Multiple Nodes

Architecture:

             ┌───────────────────┐
             │                   ↓
    START → Generate → Validate → Improve
              ↑                   │
              └───────────────────┘
                      │
                      ↓
                     END


More accurately:

             ┌→ Generate → Validate ─┐
             │                       │
             │                    Valid?
             │                    /    \
             │                  No      Yes
             │                  ↓        ↓
             └── Improve ←──────┘       END


This is useful when each iteration has multiple stages.


---

# 32. Termination Condition

Every iterative workflow should have a clear **termination condition**.

Examples:

### Condition 1 — Target reached

    number >= 10


### Condition 2 — Quality score

    score >= 0.90


### Condition 3 — Validation passed

    valid == True


### Condition 4 — Goal completed

    goal_completed == True


### Condition 5 — Maximum attempts

    attempts >= 3


### Condition 6 — No more errors

    errors == 0


A good iterative workflow knows exactly when to stop.


---

# 33. Multiple Termination Conditions

Production workflows often have more than one.

Example:

    Continue if:

    score < 0.90
    AND
    attempts < 3

Stop if:

    score >= 0.90

OR:

    attempts >= 3


Architecture:

                 ┌→ END
                 │
    Generate → Check
                 │
                 └→ Retry
                      ↓
                   Generate


---

# 34. Infinite Loop Problem

Bad design:

    Task
      ↓
    Router
      ↓
    Task
      ↓
    Router
      ↓
    Task
      ↓
    ...


If the router never returns `END`, the workflow can keep running.

### Prevention

Use:

    Maximum attempts
    Maximum iterations
    Completion condition
    Timeout
    Budget limit
    Fallback


---

# 35. Iterative Workflow and Error Recovery

Iteration is excellent for recovery.

Example:

    API Call
       ↓
    Success?
     /    \
   Yes     No
    ↓       ↓
 Continue  Retry
             ↓
          attempts?
          /      \
        Yes       No
         ↓         ↓
       API       Fallback
         ↑         ↓
         └────────END


This allows the workflow to recover automatically from temporary failures.


---

# 36. Iterative Workflow and Human-in-the-Loop

Iteration can also include human feedback.

Example:

    Generate Draft
         ↓
    Human Review
         ↓
    Approved?
      /     \
    Yes      No
    ↓         ↓
   END      Feedback
              ↓
           Improve
              ↓
         Generate Again
              ↑
              └────────


This creates a human-AI refinement loop.


---

# 37. Iterative Workflow vs Recursion

These terms are related but should not be confused.

### Iteration

Repeatedly execute a workflow based on a condition.

    A → B → A → B → END


### Recursion

A function calls itself.

    function A():
        A()


LangGraph iterative workflows are better understood as **graph cycles/loops**, not necessarily Python recursion.


---

# 38. Iterative Workflow vs Parallel Workflow

### Parallel

    START
      ↓
    ┌───┬───┐
    ↓   ↓   ↓
    A   B   C
    └───┼───┘
        ↓
       END

Goal:

> Execute independent tasks simultaneously.


### Iterative

    START
      ↓
      A
      ↓
    Check
     /   \
   Retry END
    ↓
    A

Goal:

> Repeat work until a condition is satisfied.


---

# 39. Iterative Workflow vs Conditional Workflow

Conditional:

    A
    ↓
    Router
    /   \
   B     C
   ↓     ↓
  END   END

The workflow chooses a branch.

Iterative:

    A
    ↓
    Router
    /   \
 Retry  END
   ↓
   A

The workflow chooses whether to:

    Continue the cycle

or:

    Stop.


So:

> Conditional routing chooses a path.

> Iteration uses conditional routing to repeatedly traverse a cycle.


---

# 40. Iterative Workflow + Parallel Workflow

These can also be combined.

Example:

    Generate Query
          ↓
    ┌─────┬─────┬─────┐
    ↓     ↓     ↓
   Web   RAG   SQL
    └─────┼─────┘
          ↓
       Combine
          ↓
       Evaluate
          ↓
       Good?
       /   \
     No     Yes
     ↓       ↓
  Improve   END
     ↓
 Generate Query
     ↑
     └────────


This combines:

    Sequential
    +
    Parallel
    +
    Conditional
    +
    Iterative


---

# 41. Real Agentic AI Pattern

A realistic agentic workflow might look like:

                         ┌───────────────┐
                         │               ↓
    User Goal → Agent → Decide → Tool → Observe
                  ↑                       │
                  │                       ↓
                  │                    Goal?
                  │                   /    \
                  └────── No ───────┘      Yes
                                             ↓
                                            END


This is the fundamental **agent loop**.

The agent:

1. Observes current state.
2. Decides what to do.
3. Executes an action.
4. Observes the result.
5. Decides again.
6. Stops when the goal is complete.


---

# 42. Simple Coding Pattern to Remember

When implementing an iterative workflow, remember these four pieces:

    1. Task node

    2. Condition/router

    3. Conditional edge back to task

    4. Conditional edge to END


Conceptually:

    graph.add_node("task", task)

    graph.add_conditional_edges(
        "task",
        router,
        {
            "retry": "task",
            "done": END
        }
    )


This is the core implementation pattern.


---

# 43. Best Practices

## 1. Always define a stopping condition

Never create a loop without knowing how it ends.


## 2. Add a maximum retry/iteration limit

Protect against unexpected behavior.


## 3. Keep iteration state explicit

Track:

    attempts
    score
    validation
    feedback
    errors


## 4. Keep the router simple

The router should decide:

    retry

or:

    done

rather than perform the actual work.


## 5. Preserve useful feedback

If validation fails, store the reason:

    feedback = "Missing budget information."


The next iteration can use it to improve the output.


## 6. Avoid unnecessary LLM calls

If the result is already good enough, terminate early.


## 7. Test termination conditions

Test:

    Immediate success
    One retry
    Multiple retries
    Maximum retries
    Failure/fallback


---

# 44. Common Mistakes

### Mistake 1 — No termination condition

Can create an infinite loop.


### Mistake 2 — No retry limit

An LLM may repeatedly produce an invalid result.


### Mistake 3 — Losing state

The next iteration may need previous output or feedback.


### Mistake 4 — Incorrect routing

A router can accidentally keep sending the workflow back to the same node.


### Mistake 5 — Iterating when unnecessary

Don't use a loop when the task can be completed deterministically in one pass.


### Mistake 6 — Confusing iteration with parallelism

Iteration means:

    Repeat

Parallelism means:

    Execute independent work concurrently.


---

# 45. Interview Questions

### Q1. What is an iterative workflow in LangGraph?

An iterative workflow repeatedly executes one or more nodes until a specified termination condition is met.

### Q2. How is iteration implemented in LangGraph?

Typically using graph cycles and conditional edges that route execution back to an earlier node or to `END`.

### Q3. What is a self-loop?

A graph transition where a node routes back to itself.

Example:

    task → task


### Q4. Why is state important?

State stores information such as outputs, attempts, scores, feedback, and validation results across iterations.

### Q5. How do you prevent infinite loops?

Use termination conditions, maximum attempts, timeouts, or fallback paths.

### Q6. Can iterative workflows use LLMs?

Yes. They are commonly used for generation → evaluation → refinement loops.

### Q7. Can iterative workflows be used in Agentic AI?

Yes. The agent loop itself is fundamentally iterative.

### Q8. What is the difference between iteration and conditional routing?

Conditional routing selects a path based on a condition. Iteration uses conditional routing to repeatedly execute a cycle until a stopping condition is satisfied.

### Q9. Can parallel and iterative workflows be combined?

Yes. A workflow can perform parallel tasks during each iteration and repeat the process if validation fails.

### Q10. Give a real-world iterative workflow example.

Code generation:

    Generate Code
        ↓
    Run Tests
        ↓
    Tests Passed?
      /       \
    Yes        No
    ↓           ↓
   END       Fix Code
               ↓
           Run Tests Again


---

# 46. Quick Revision

## Definition

> **Iterative Workflow = A workflow that repeatedly executes one or more nodes until a termination condition is satisfied.**


## Basic Pattern

    START
      ↓
    Task
      ↓
    Check
     /   \
   Retry END
     ↓
    Task
     ↑
     └────


## Core Components

    State
    +
    Node
    +
    Condition/Router
    +
    Conditional Edge
    +
    Cycle
    +
    Termination Condition


## Important Concepts

- Loop
- Cycle
- Self-loop
- Router
- Conditional edge
- State evolution
- Retry
- Validation
- Refinement
- Maximum attempts
- Termination
- Fallback


---

# 47. Three Important Workflow Patterns

## Sequential

    A → B → C → END

**Fixed order**

> "Do these steps one after another."


## Parallel

    ┌→ A ─┐
    ├→ B ─┼→ D → END
    └→ C ─┘

**Independent execution**

> "Do these independent tasks together."


## Conditional

    A
    ↓
    Router
    /   \
   B     C

**Dynamic routing**

> "Choose the appropriate path."


## Iterative

    A
    ↓
    Router
    /   \
   A    END

**Repeated execution**

> "Keep doing the work until the condition is satisfied."


---

# 48. Combined LangGraph Mental Model

Modern AI systems often combine all four:

                         ┌→ Task A ─┐
                         │          │
    START → Router ──────┼→ Task B ─┼→ Combine
                         │          │
                         └→ Task C ─┘
                                      ↓
                                   Validate
                                      ↓
                                    Good?
                                   /    \
                                 No      Yes
                                 ↓        ↓
                              Improve    END
                                 ↓
                              Router
                                 ↑
                                 └────────


This architecture can contain:

    Conditional routing
          +
    Parallel execution
          +
    Sequential processing
          +
    Iteration
          +
    Validation


---

# 49. Final Mental Model

Think of an editor reviewing a document.

    Write Draft
        ↓
      Review
        ↓
    Is it good?
      /     \
    No       Yes
    ↓         ↓
   Edit      Publish
    ↓         ↓
   Review    END
     ↑
     └────────


The editor doesn't publish after the first draft.

Instead:

    Draft
      ↓
    Review
      ↓
    Improve
      ↓
    Review
      ↓
    Improve
      ↓
    Review
      ↓
    Good enough
      ↓
    Publish


That is an **Iterative Workflow**.

### One-line memory trick

> **Sequential = Go forward.**
>
> **Parallel = Split and execute.**
>
> **Conditional = Choose a path.**
>
> **Iterative = Repeat until done.**

### Most Important LangGraph Pattern

    Task
      ↓
    Condition
      ↓
    ┌─────────────┐
    ↓             ↓
   Retry          END
    ↓
   Task

### Agentic AI Connection

    Observe
       ↓
    Reason
       ↓
     Act
       ↓
    Observe Result
       ↓
    Goal Complete?
      /        \
    No          Yes
    ↓            ↓
   Act          END
    ↑
    └────────────

Therefore:

> **Iteration is one of the fundamental building blocks of Agentic AI because an agent normally needs multiple observe → decide → act cycles before completing a goal.**